# 5.7 — FastTD3 directional walk from scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/tutorial/fasttd3-directional-scratch/tutorials/5.7_FastTD3_Directional_From_Scratch.ipynb)

Train `myoFullBodyDirectional-v0` (528 → 354) with FastTD3. No MuscleMimic teacher.

1. **CPU smoke** (default) — a few hundred steps. Confirms the loop before you spend GPU.
2. Runtime → GPU, set `SMOKE = False`, run **train**.

Sim is CPU MuJoCo. GPU only speeds the TD3 updates (AMP + larger batch). No Hugging Face token: this run does not download MuscleMimic motion clips.

## Install

In [ ]:
from __future__ import annotations

import os; _nb = globals().get("__vsc_ipynb_file__"); _nb and os.chdir(os.path.dirname(_nb))

import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

GIT_URL = os.environ.get("MYOSUITE_GIT_URL", "https://github.com/MyoHub/myosuite.git")
GIT_REF = os.environ.get("MYOSUITE_GIT_REF", "tutorial/fasttd3-directional-scratch")

if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")


def _repo_root() -> Path | None:
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "myosuite" / "__init__.py").is_file():
            return d
    return None


if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "libegl1", "libgles2"], check=False)
    pkgs = ["imageio[ffmpeg]", "mediapy"]
    try:
        import torch
        if not torch.cuda.is_available():
            pkgs.append("torch")
    except ImportError:
        pkgs.append("torch")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

repo = _repo_root()
if repo is None:
    dest = Path("/content/myosuite") if IN_COLAB else Path.cwd() / "myosuite_src"
    if not dest.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(dest)])
    else:
        subprocess.check_call(["git", "-C", str(dest), "fetch", "--depth", "1", "origin", GIT_REF])
        subprocess.check_call(["git", "-C", str(dest), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
    if IN_COLAB:
        os.chdir(dest)
    if str(dest) not in sys.path:
        sys.path.insert(0, str(dest))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{dest}[musclemimic]"])
    repo = dest
else:
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

HELPER = repo / "tutorials" / "files" / "5.7"
_nb_dir = Path(_nb).parent if _nb else Path.cwd()
for cand in (
    HELPER / "fasttd3_directional.py",
    _nb_dir / "files" / "5.7" / "fasttd3_directional.py",
    Path.cwd() / "files" / "5.7" / "fasttd3_directional.py",
):
    if cand.is_file():
        HELPER = cand.parent
        break
else:
    raise FileNotFoundError(
        "Need tutorials/files/5.7/fasttd3_directional.py next to this notebook "
        "(open it from the myosuite clone, not a lone Colab upload)."
    )
if str(HELPER) not in sys.path:
    sys.path.insert(0, str(HELPER))

print("IN_COLAB:", IN_COLAB, " repo:", repo, " helper:", HELPER)

## Knobs

Leave `SMOKE = True` and a **CPU** runtime until the next cell prints `ok`.
Then: Runtime → Change runtime type → T4/A100, set `SMOKE = False`, re-run from install.

In [ ]:
# @title FastTD3 knobs
SMOKE = True  # @param {type:"boolean"}
TRAIN_STEPS = 200_000  # @param {type:"integer"}
N_ENVS = 8  # @param {type:"integer"}
BATCH = 1024  # @param {type:"integer"}

from fasttd3_directional import eval_dirs, get_device, render_east, train

device = get_device(smoke=SMOKE)
OUT = Path("runs/fasttd3_scratch")
steps = 400 if SMOKE else TRAIN_STEPS
n_envs = 1 if SMOKE else N_ENVS
batch = 64 if SMOKE else BATCH
print(f"SMOKE={SMOKE}  device={device}  steps={steps}  envs={n_envs}  batch={batch}")
if not SMOKE and device.type == "cpu":
    raise SystemExit("No GPU. Switch the runtime to T4/A100, or keep SMOKE=True.")

## Smoke / train

In [ ]:
actor = train(
    steps=steps, n_envs=n_envs, batch=batch, device=device, out=OUT, smoke=SMOKE
)
summary = eval_dirs(
    actor, device, labels=["E", "N"] if SMOKE else None, steps=40 if SMOKE else 120
)
print("summary", {k: round(v, 3) for k, v in summary.items()})
if SMOKE:
    print("ok — CPU loop works. Flip SMOKE=False on a GPU runtime for training.")
else:
    mp4 = render_east(actor, device, OUT / "east.mp4")
    try:
        from IPython.display import Video, display
        display(Video(str(mp4), embed=True, width=420))
    except Exception:
        print(mp4)